#### Setting environment variable

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if GROQ_API_KEY:
    print("GROQ_API_KEY loaded")
else:
    print("GROQ_API_KEY is null")

GROQ_API_KEY loaded


#### Data Ingestion

In [2]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import FAISS

In [ ]:
PDF_PATH = "../data/attention.pdf"
loader = PyPDFLoader(str(PDF_PATH))

# if want to read all PDFs in a folder
# loader = DirectoryLoader(str(PDF_DIR), loader_cls = PyPDFLoader)

documents = loader.load()

print(f"Loaded {len(documents)} page(s) from {PDF_PATH}")

Loaded 15 page(s) from ../data/attention.pdf


In [7]:
documents[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': '../data/attention.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}

In [8]:
print(documents[0].page_content)

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exp

#### Chunking of the Data

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20
)

text_chunks = text_splitter.split_documents(documents)
print(f"Created {len(text_chunks)} chunks")

Created 251 chunks


In [11]:
[(chunk.metadata.get("page"), chunk.page_content[:120]) for chunk in text_chunks[:3]]

[(0,
  'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this pap'),
 (0,
  'Ashish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research'),
 (0,
  'Llion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser')]

In [12]:
sample_index = min(50, len(text_chunks) - 1)
print(text_chunks[sample_index].page_content)

wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is


#### Vector DB Configuration

In [13]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

vector_store = FAISS.from_documents(text_chunks, embedding)
vector_store.save_local("../data/faiss_index")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 26836.41it/s]


In [18]:
vector_store

#### Data Retrieval

In [19]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True}
)

# Load FAISS
vector_store = FAISS.load_local(
    "../data/faiss_index",
    embeddings=embedding,
    allow_dangerous_deserialization=True
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 33619.78it/s]


In [21]:
docs = vector_store.similarity_search(
    "What is attention?",
    k=3
)

for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
--------------------------------------------------
Document 2:
values.
In practice, we compute the attention function on a set of queries simultaneously, packed together
--------------------------------------------------
Document 3:
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
--------------------------------------------------


In [23]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 4}
)

#### Data Generation

Preparing Prompt

In [24]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [25]:
prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [26]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [27]:
output_parser = StrOutputParser()

Loading llm model

In [35]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

for model in client.models.list().data:
    print(model.id)

allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
whisper-large-v3
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3-32b
groq/compound-mini
llama-3.3-70b-versatile
groq/compound
qwen/qwen3.6-27b
llama-3.1-8b-instant
meta-llama/llama-4-scout-17b-16e-instruct
canopylabs/orpheus-v1-english


In [37]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

Configuring LLM Chain

In [39]:
chain_inputs = {"context": retriever | format_docs, "question": RunnablePassthrough()}

rag_chain = chain_inputs | prompt | llm | output_parser

In [40]:
rag_chain.invoke("tell me about the phases in self attention")

'Self‑attention works in three main phases.  \n\n1. **Projection** – each input token is linearly transformed into three vectors: a query\u202f\\(Q\\), a key\u202f\\(K\\), and a value\u202f\\(V\\). These projections are learned parameters and are applied independently to every position.  \n\n2. **Scoring & Normalisation** – for every pair of positions the dot‑product of the query of one token with the key of another is computed, scaled (by \\(\\sqrt{d_k}\\)), and then passed through a softmax. This yields attention weights that sum to\u202f1 across all tokens, indicating how much each token should attend to every other token.  \n\n3. **Aggregation** – each token’s output is the weighted sum of the value vectors of all tokens, using the attention weights from the previous step.  \n\nWhen multiple heads are used, the three phases are performed in parallel on different sub‑spaces, and their results are concatenated and linearly projected again. In practice, the first two phases let the mo